In [4]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

  Using cached pypdf-6.14.2-py3-none-any.whl.metadata (7.2 kB)
  Using cached pymupdf-1.28.0-cp310-abi3-win_amd64.whl.metadata (26 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached pyproject_hooks-1.2.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached durationpy-0.10-py3-none-any.whl.metadata (340 bytes)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached oauthlib-3.3.1-py3-none-any.whl.metadata (7.9 kB)
   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
    --------------------------------------- 0.3/19.8 MB ? eta -:--:--
   - -------------------------------------- 0.5/19.8 MB 1.2 MB/s eta 0:00:16
   -- ------

  You can safely remove it manually.
  You can safely remove it manually.


In [1]:
from langchain_core.documents import Document

In [2]:
sample_docs = Document(
    page_content="Hello World @",
    metadata = {"source" : "http//www.google.com"}
)

In [3]:
sample_docs

Document(metadata={'source': 'http//www.google.com'}, page_content='Hello World @')

In [4]:
# Text data
from langchain_community.document_loaders.text import TextLoader

C:\Users\Hemant\AppData\Local\Temp\ipykernel_27564\3188525453.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [5]:
loader = TextLoader(
    "Data/Python.txt",
    encoding = "utf-8"
)

In [6]:
document = loader.load()

In [7]:
document

[Document(metadata={'source': 'Data/Python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

In [8]:
# PDF Data
# from langchain_community.document_loaders.pdf import PyPDFLoader

# from langchain_community.document_loaders.pdf import PyMuPDFLoader

In [9]:
# pdf_loader = PyPDFLoader(
#     "Data/research.pdf"
# )

In [10]:
# document = pdf_loader.load()

In [11]:
# document

## Ingestion Pipeline

In [12]:
# Data => Document

import os
from langchain_community.document_loaders.pdf import PyPDFLoader

### Document

In [13]:
# pdf => docs conversion function
def load_all_pdfs():
    folder_path = "Data/pdfs"
    docs = []
    num_of_docs = 0

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete pdf name
            pdf_path = os.path.join(folder_path , filename)
            
            pdf_loader = PyPDFLoader(pdf_path)

            doc = pdf_loader.load()

            docs.extend(doc)

            num_of_docs += 1

    print(f" number of docs : {num_of_docs}")
    print(f"total pages : {len(docs)}")
    return docs

In [14]:
all_pdf_docs = load_all_pdfs()

 number of docs : 2
total pages : 32


In [15]:
# Chunks

!pip install langchain_text_splitters

### Chunks

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(document , chunk_size = 500 , chunk_overlap = 50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_documents = text_splitter.split_documents(document)
    return chunked_documents

In [17]:
chunks = split_docs(all_pdf_docs)

In [18]:
type(all_pdf_docs)

list

In [19]:
len(chunks)

321

### Embedding

In [20]:
from sentence_transformers import SentenceTransformer

In [35]:
class EmbeddingManager :
    def __init__(self , model_name = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        print(f"loading model ...  {self.model_name}")
        self.model = SentenceTransformer(self.model_name)
        print(f"embedding dimensions : {self.model.get_embedding_dimension()}")

    def generate_embedding(self , text):
        embeddings = self.model.encode(text , show_progress_bar = True)
        print("embedding shape : ",embeddings.shape)
        return embeddings

In [36]:
embedding = EmbeddingManager()

loading model ...  all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions : 384


### Vector DB

In [37]:
import chromadb
import uuid

In [38]:
class VectorStoreManager:
    def __init__(self , persist_directory = "data/vector_store" , collection_name = "pdf_documents"):
        self.persist_directory = persist_directory
        self.collection_name = collection_name 
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory , exist_ok = True)

        self.client = chromadb.PersistentClient(path = self.persist_directory)

        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata = {"description" : "vector store collection for pdf embeddings in docs"}
        )

        print("initialized the vector store with collection : " , self.collection_name)
        print("docs in collections : " , self.collection.count())

    def add_document(self , documents , embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("len of docs is not equal to len of embeddings")

        # id , embedding , docs , metadata
        ids = []
        docs_content = []
        all_metadata = []
        embeddings_list = []

        for i , (doc , embedding) in enumerate(zip(documents , embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            all_metadata.append(metadata)

            docs_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids = ids,
                metadatas = all_metadata,
                documents = docs_content,
                embeddings = embeddings_list
            )

        print(f"total documents added in collections : {len(docs_content)}")
        print("docs in collections : " , self.collection.count())

In [39]:
vector_store = VectorStoreManager()

initialized the vector store with collection :  pdf_documents
docs in collections :  0


In [40]:
text = [doc.page_content for doc in chunks]

embeddings = embedding.generate_embedding(text)

vector_store.add_document(chunks , embeddings)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embedding shape :  (321, 384)
total documents added in collections : 321
docs in collections :  321


### Retrieve

In [41]:
from sklearn.metrics.pairwise import cosine_similarity

In [58]:
class RAGRetriever:
    def __init__(self , embeddings_manager , vector_store):
        self.embeddings_manager = embeddings_manager
        self.vector_store = vector_store

    def retrieve(self , query , top_k = 5 , threshold = 0.0):
        query_embeddings = self.embeddings_manager.generate_embedding([query])[0]

        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k
        )

        retrieved_docs = []

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i , (doc_id , metadata , document , distance) in enumerate(zip(docs_id , metadata , document , distance)):
                similarity_score = 1 - distance

                if similarity_score >= threshold:
                    retrieved_docs.append({
                        "id" : doc_id,
                        "metadatas" : metadatas,
                        "documents" : documents,
                        "distances" : distances,
                        "similarity_score" : similarity_score,
                        "rank" : i+1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else :
            print("no document")

        return retrieved_docs

In [59]:
rag_retriever = RAGRetriever(embedding , vector_store)

In [60]:
rag_retriever.retrieve("What is RAG?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape :  (1, 384)


NameError: name 'docs_id' is not defined